# 02 - Return times and extremal index

Research question: which generators show faster recurrence of tail events and which are closer to the exponential return-time benchmark?

Data dependencies: none. This notebook uses only synthetic simulations.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from dynsys_econometrics.extremes import estimate_extremal_index
from dynsys_econometrics.return_times import (
    compute_return_times,
    empirical_survival_curve,
    ks_exponential_diagnostic,
)
from dynsys_econometrics.simulation import simulate_ar1, simulate_garch11, simulate_iid_gaussian, simulate_pomeau_manneville


In [ ]:
n_steps = 4000
threshold_quantile = 0.95

paths = {
    "iid": simulate_iid_gaussian(n_steps, seed=13),
    "ar1": simulate_ar1(n_steps, phi=0.8, sigma=1.0, seed=13),
    "garch11": simulate_garch11(n_steps, seed=13),
    "pomeau_manneville": simulate_pomeau_manneville(n_steps, x0=2e-3, a=1.0, z=1.5),
}

rows = []
for name, values in paths.items():
    threshold = float(np.quantile(values, threshold_quantile))
    return_times = compute_return_times(values, threshold=threshold)
    ks_stat, ks_pvalue, reject_null, _ = ks_exponential_diagnostic(values, threshold=threshold)
    theta = estimate_extremal_index(values, threshold_quantile=threshold_quantile, run_length=3)
    rows.append(
        {
            "series": name,
            "n_return_times": int(return_times.size),
            "mean_return_time": float(return_times.mean()) if return_times.size else float("nan"),
            "theta": theta,
            "ks_stat": ks_stat,
            "ks_pvalue": ks_pvalue,
            "reject_exponential": reject_null,
        }
    )

summary = pd.DataFrame(rows)
summary


In [ ]:
example = paths["ar1"]
threshold = float(np.quantile(example, threshold_quantile))
example_return_times = compute_return_times(example, threshold=threshold)
times, survival = empirical_survival_curve(example_return_times)

output_dir = Path("article/figures")
output_dir.mkdir(parents=True, exist_ok=True)

fig, ax = plt.subplots(figsize=(8, 4))
if times.size > 0:
    ax.plot(times, survival, marker="o", color="#f58518")
else:
    ax.text(0.5, 0.5, "No return times available", ha="center", va="center")
ax.set_title("Empirical survival of return times (AR1)")
ax.set_xlabel("return time")
ax.set_ylabel("S(t)")
fig.tight_layout()
fig.savefig(output_dir / "02_return_times_and_extremal_index.png", dpi=160)
plt.close(fig)
str(output_dir / "02_return_times_and_extremal_index.png")
